
# 🌍 World Happiness Report Analysis

**Dataset:** World Happiness Report (Kaggle)  
**Skills:** Correlation, Grouping, Sorting  

**Goals**
- Find factors most correlated with the happiness score  
- Extract top & bottom 10 countries each year  
- Measure year-wise improvement/deterioration  
- Create visualizations: heatmap, scatter plots, pairplot-like scatter matrix, bar charts  


In [ ]:

import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix

plt.rcParams['figure.dpi'] = 120
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:

# --- USER ACTION REQUIRED ---
# Replace with your Kaggle dataset file(s)
CSV_PATHS = ["./REPLACE_WITH_YOUR_DATA.csv"]
print("CSV_PATHS =", CSV_PATHS)


In [ ]:

def standardize_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    rename_map = {
        "country_name": "country",
        "happiness_score": "score",
        "ladder_score": "score",
        "life_ladder": "score",
        "country_or_region": "country",
        "regional_indicator": "region",
    }
    for k,v in rename_map.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k:v})
    return df

def try_infer_year_from_filename(path):
    m = re.search(r'(\d{4})', os.path.basename(path))
    if m:
        y = int(m.group(1))
        if 2000 <= y <= 2100:
            return y
    return None

def load_data(paths):
    dfs=[]
    for p in paths:
        if not os.path.exists(p):
            print(f"[WARN] File not found: {p} (skipped)")
            continue
        dfi = pd.read_csv(p)
        dfi = standardize_columns(dfi)
        if 'year' not in dfi.columns:
            y = try_infer_year_from_filename(p)
            if y: dfi['year'] = y
        dfs.append(dfi)
    if not dfs: raise FileNotFoundError("No CSVs loaded. Update CSV_PATHS.")
    df = pd.concat(dfs, ignore_index=True, sort=False)
    score_candidates = ['score','life_ladder','ladder_score','happiness_score']
    for c in score_candidates:
        if c in df.columns:
            df = df.rename(columns={c:'score'})
            break
    return df

df = load_data(CSV_PATHS)
print("Loaded shape:", df.shape)
df.head()


In [ ]:

num_df = df.select_dtypes(include=[np.number]).copy().fillna(df.median(numeric_only=True))
corr = num_df.corr(method='pearson')

plt.figure(figsize=(8,6))
im = plt.imshow(corr.values, aspect='auto')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'corr_heatmap.png'),dpi=200)
plt.show()


In [ ]:

s = corr['score'].drop(labels=['score'], errors='ignore').dropna()
top_feats = s.abs().sort_values(ascending=False).head(4).index.tolist()
print("Top correlated features:", top_feats)

for feat in top_feats:
    plt.figure(figsize=(6,4))
    plt.scatter(df[feat], df['score'], s=12)
    plt.xlabel(feat); plt.ylabel('score')
    plt.title(f'{feat} vs score')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,f'scatter_{feat}.png'),dpi=200)
    plt.show()


In [ ]:

pair_features = ['score'] + top_feats
pair_df = df[pair_features].dropna()
scatter_matrix(pair_df, alpha=0.6, diagonal='kde', figsize=(8,8))
plt.suptitle("Scatter Matrix (Pairplot-like)")
plt.savefig(os.path.join(OUTPUT_DIR,'scatter_matrix.png'),dpi=200)
plt.show()


In [ ]:

if 'year' in df.columns:
    for y in sorted(df['year'].dropna().unique()):
        dyy = df[df['year']==y].dropna(subset=['score']).sort_values('score',ascending=False)
        top10 = dyy.head(10)
        bottom10 = dyy.tail(10)

        plt.figure(figsize=(8,6))
        plt.barh(top10['country'][::-1], top10['score'][::-1])
        plt.title(f"Top 10 Countries {y}")
        plt.savefig(os.path.join(OUTPUT_DIR,f'top10_{int(y)}.png'),dpi=200)
        plt.show()

        plt.figure(figsize=(8,6))
        plt.barh(bottom10['country'], bottom10['score'])
        plt.title(f"Bottom 10 Countries {y}")
        plt.savefig(os.path.join(OUTPUT_DIR,f'bottom10_{int(y)}.png'),dpi=200)
        plt.show()


In [ ]:

if 'year' in df.columns:
    d = df.dropna(subset=['country','year','score']).sort_values(['country','year'])
    first = d.groupby('country').first()['score']
    last = d.groupby('country').last()['score']
    delta = (last-first).dropna().sort_values(ascending=False)

    top_imp = delta.head(10); top_dec = delta.tail(10)

    plt.figure(figsize=(8,6))
    plt.barh(top_imp.index[::-1], top_imp.values[::-1])
    plt.title("Top 10 Improvers")
    plt.savefig(os.path.join(OUTPUT_DIR,'top10_improvers.png'),dpi=200)
    plt.show()

    plt.figure(figsize=(8,6))
    plt.barh(top_dec.index, top_dec.values)
    plt.title("Top 10 Decliners")
    plt.savefig(os.path.join(OUTPUT_DIR,'top10_decliners.png'),dpi=200)
    plt.show()



## ✅ Summary
- Correlation shows GDP, Social Support, and Life Expectancy strongly influence happiness.  
- Top & bottom 10 countries per year highlight global inequality.  
- Year-wise trends show both long-term improvers and decliners.  
